In [ ]:
import ee
import time

In [ ]:
india = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq('country_na', 'India')).geometry()

In [ ]:
# 3. Temporal Parameters
start_year = 2019
end_year = 2023
total_months = (end_year - start_year + 1) * 12

In [ ]:
# 5. Loop through all 60 months and compile the data
# This creates a massive FeatureCollection (Data Table)
sequence = ee.List.sequence(0, total_months - 1)
all_data = ee.FeatureCollection(sequence.map(get_monthly_data)).flatten()

In [ ]:
while task.active():
    print(f"Status: {task.status()['state']}... (Checking again in 10 seconds)")
    time.sleep(10)

print(f"🎉 TASK FINISHED! Final Status: {task.status()['state']}")

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np

# This will ask you to click a link and allow Colab to read your Drive
drive.mount('/content/drive')

# Load the shiny new dataset!
# (Make sure this path matches where GEE saved it. It should be in the 'Drought_Project' folder)
file_path = '/content/drive/MyDrive/Drought_Project/Pan_India_Multi_Drought_Dataset.csv'
df = pd.read_csv(file_path)

print(f"✅ Successfully loaded {len(df):,} rows of Pan-India climate data!")
print(df.head())

Mounted at /content/drive
✅ Successfully loaded 331,500 rows of Pan-India climate data!
   latitude  longitude  year  month  precipitation  temperature  \
0  7.074233  93.761658  2019      1      66.113237    26.647850   
1  7.972548  93.537079  2019      1      30.468192    27.238486   
2  8.197127  77.367404  2019      1       2.177982    26.227478   
3  8.197127  77.591983  2019      1       1.268941    25.760018   
4  8.197127  93.537079  2019      1      34.778899    27.422669   

   soil_moisture     ndvi  groundwater  
0       0.241146  0.71545     5.167862  
1       0.221371  0.74185    24.461311  
2       0.151351  0.65650   -16.443778  
3       0.115543  0.59145   -16.443778  
4       0.217448  0.64450    24.461311  


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

print("⚙️ Preparing the Data...")

# 1. Calculate the Z-Score (Drought Index) based on historical rainfall
# We group by 'month' so the AI knows that a dry July is worse than a dry January!
df['rain_mean'] = df.groupby('month')['precipitation'].transform('mean')
df['rain_std'] = df.groupby('month')['precipitation'].transform('std')
df['z_score'] = (df['precipitation'] - df['rain_mean']) / df['rain_std']

# 2. Apply your 10-Level Classification
def classify_10(val):
    if np.isnan(val): return np.nan
    if val >= 2.0: return 1.0         # Extreme Wet
    elif val >= 1.6: return 2.0
    elif val >= 1.3: return 3.0
    elif val >= 0.8: return 4.0
    elif val >= 0.5: return 5.0
    elif val >= -0.5: return 6.0      # Normal / Neutral
    elif val >= -0.8: return 7.0
    elif val >= -1.3: return 8.0      # Moderate Drought
    elif val >= -1.6: return 9.0      # Severe Drought
    else: return 10.0                 # Extreme Drought

df['drought_class'] = df['z_score'].apply(classify_10)

# 3. Select Features (X) and Target (y)
# Notice we are now feeding it 5 completely different dimensions of climate data!
features = ['latitude', 'longitude', 'month', 'temperature', 'soil_moisture', 'ndvi', 'groundwater', 'precipitation']
df_clean = df.dropna(subset=features + ['drought_class'])

X = df_clean[features]
y = df_clean['drought_class']

# 4. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"🌲 Training Pan-India Random Forest on {len(X_train):,} samples...")
print("(This might take 2-5 minutes because the dataset is HUGE!)")

# 5. Train the Model (With Class Balancing for rare extreme droughts)
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 6. Evaluate and Save
print("✅ Training Complete! Testing accuracy...")
y_pred = rf_model.predict(X_test)
print("-" * 50)
print(classification_report(y_test, y_pred, zero_division=0))
print("-" * 50)

# Save the Master Model!
joblib.dump(rf_model, 'pan_india_drought_model.joblib')
print("💾 MASTER MODEL SAVED as 'pan_india_drought_model.joblib'!")

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from shapely.geometry import Point
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

# --- CONFIGURATION ---
districts_excel_path = '/content/drive/MyDrive/Drought_Project/India_Districts.xlsx'
district_spei_folder = '/content/drive/MyDrive/Drought_Project/Districts/'
shapefile_path = '/content/drive/MyDrive/Drought_Project/Administrative-Boundaries_Shapefile/DISTRICT_BOUNDARY.shp'
gee_csv_path = '/content/drive/MyDrive/Drought_Project/Pan_India_Multi_Drought_Dataset.csv'

print("📂 Step 1: Loading District Metadata...")
df_dist_info = pd.read_excel(districts_excel_path)

# Based on your image: Column 0 is 'ID', Column 1 is 'District'
id_col = 'ID'
name_col = 'District'

print("🧵 Step 2: Stitching SPEI Text Files...")
all_district_data = []

for index, row in df_dist_info.iterrows():
    dist_id = int(row[id_col])
    dist_name = row[name_col]

    # Matches format: data_distwise_id_1, data_distwise_id_2, etc.
    file_name = f"data_distwise_id_{dist_id}"
    file_path = os.path.join(district_spei_folder, file_name)

    if os.path.exists(file_path):
        # The README says 5 columns: Year Month SPEI-1month SPEI-4month SPEI-12month
        temp_df = pd.read_csv(file_path, sep='\s+', header=None,
                              names=['Year', 'Month', 'SPEI_1', 'SPEI_4', 'SPEI_12'])
        temp_df['District_Name_Atlas'] = dist_name
        all_district_data.append(temp_df)

df_atlas = pd.concat(all_district_data, ignore_index=True)
print(f"✅ Loaded {len(df_atlas):,} monthly records from the Atlas.")

print("🗺️ Step 3: Spatial Join (Pixels to Districts)...")
df_gee = pd.read_csv(gee_csv_path)
gdf_districts = gpd.read_file(shapefile_path)

# Convert GEE CSV to Spatial GeoDataFrame
geometry = [Point(xy) for xy in zip(df_gee['longitude'], df_gee['latitude'])]
gdf_gee = gpd.GeoDataFrame(df_gee, geometry=geometry, crs="EPSG:4326")
gdf_districts = gdf_districts.to_crs("EPSG:4326")

# Join: Adds district info to every satellite pixel
# Note: Check if your shapefile uses 'DISTRICT' or 'NAME' for the district column
gdf_mapped = gpd.sjoin(gdf_gee, gdf_districts, how="inner", predicate="within")

print("🔗 Step 4: Final Data Fusion...")
# Standardize column names for the merge
gdf_mapped = gdf_mapped.rename(columns={'year': 'Year', 'month': 'Month'})

# We merge based on the District Name from the Shapefile matching the Atlas
# Replace 'DISTRICT' with the actual column name from your shapefile (usually 'DISTRICT')
final_df = pd.merge(gdf_mapped, df_atlas, left_on=['District', 'Year', 'Month'],
                    right_on=['District_Name_Atlas', 'Year', 'Month'], how='inner')

print("⚖️ Step 5: Classification & Training...")

def classify_spei(val):
    if val > -0.5: return 0  # Normal/Wet
    if val > -1.0: return 1  # Mild/Abnormal Dry
    if val > -1.5: return 2  # Moderate Drought
    if val > -2.0: return 3  # Severe Drought
    return 4                 # Extreme/Exceptional Drought

final_df['label'] = final_df['SPEI_1'].apply(classify_spei)

features = ['precipitation', 'temperature', 'soil_moisture', 'ndvi', 'groundwater']
df_ml = final_df.dropna(subset=features + ['label'])

X = df_ml[features]
y = df_ml['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

print("\n📊 FINAL PERFORMANCE REPORT:")
print(classification_report(y_test, rf_model.predict(X_test)))

joblib.dump(rf_model, 'pan_india_drought_model_atlas.joblib')
print("💾 MODEL SAVED! You are ready to present.")

<>:37: SyntaxWarning: invalid escape sequence '\s'
<>:37: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1242/3605656912.py:37: SyntaxWarning: invalid escape sequence '\s'
  temp_df = pd.read_csv(file_path, sep='\s+', header=None,


📂 Step 1: Loading District Metadata...
🧵 Step 2: Stitching SPEI Text Files...
✅ Loaded 1,054,152 monthly records from the Atlas.
🗺️ Step 3: Spatial Join (Pixels to Districts)...
🔗 Step 4: Final Data Fusion...
⚖️ Step 5: Classification & Training...

📊 FINAL PERFORMANCE REPORT:
              precision    recall  f1-score   support

           0       0.87      0.98      0.92     19013
           1       0.76      0.47      0.58      3383
           2       0.74      0.56      0.63      2406
           3       0.71      0.52      0.60      1189
           4       0.81      0.66      0.73       577

    accuracy                           0.85     26568
   macro avg       0.78      0.64      0.69     26568
weighted avg       0.84      0.85      0.83     26568

💾 MODEL SAVED! You are ready to present.
